In [52]:
from pathlib import Path
import gcamreader
import os
import pandas as pd
import numpy as np
from utils import convert_to_mt, ej_to_twh
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [53]:
ej_to_twh(0.554)

153.88888901200002

In [54]:
for t in range(5, 21, 5):
    f = 1 / (1 + np.exp(0.1 * (t-22.5)))
    print(t, f)

5 0.8519528019683106
10 0.7772998611746911
15 0.679178699175393
20 0.5621765008857981


In [55]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4':    28,
    'N2O':    265,
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

def ej_to_twh(ej):
    """
    Convert energy from exajoules (EJ) to terawatt-hours (TWh).

    Parameters:
    ej (float): Energy in exajoules.

    Returns:
    float: Energy in terawatt-hours.
    """
    twh = ej * 277.777778
    return twh

# Define custom colors for each class
custom_colors = {
    'Solar': '#FECB52',  # Yellow
    'Wind': 'rgb(136,204,238)',  # Light blue
    'Hydro': 'rgb(95, 70, 144)',  # Dark blue
    'Nuclear': '#AB63FA',  # Orange
    'Biomass': 'rgb(115, 175, 72)',  # Dark green
    'Gas w/ CCS': '#DEA0FD',     # Pink
    'Gas': '#FFA15A',    # Purple
    'Coal w/ CCS': '#750D86',    # Maroon
    'Coal': '#222A2A',   # Red
    'Others': 'rgb(217,217,217)',      # Dark orange
    'Hydrogen': "#727DCD",
    'Ammonia': "rgb(231,63,116)",
    'Oil': "#7D1215",
    'Oil w/ CCS': "#8D5757",
}

In [56]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [57]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250715_2"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Enhanced-Ambition


In [58]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Enhanced-Ambition']

In [59]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [60]:
df = conn.runQuery(queries[21], scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
# df['vint'] = df['technology'].str.split('=').str[1].astype(int)
df.head()

,Units,scenario,region,sector,subsector,Year,technology,value
0,None Specified,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,2020,rooftop_pv,1.0
1,None Specified,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,2025,rooftop_pv,1.0
2,None Specified,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,2030,rooftop_pv,1.0
3,None Specified,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,2035,rooftop_pv,1.0
4,None Specified,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,2040,rooftop_pv,1.0


In [61]:
df[(df['Year'] == 2025)]

,Units,scenario,region,sector,subsector,Year,technology,value
1,None Specified,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,2025,rooftop_pv,1.000000
21,None Specified,Enhanced-Ambition,South Korea,electricity,biomass,2025,biomass (IGCC),1.000000
22,None Specified,Enhanced-Ambition,South Korea,electricity,biomass,2025,biomass (conv),1.000000
78,None Specified,Enhanced-Ambition,South Korea,electricity,coal,2025,coal (IGCC CCS),1.000000
79,None Specified,Enhanced-Ambition,South Korea,electricity,coal,2025,coal (conv pul CCS),1.000000
140,None Specified,Enhanced-Ambition,South Korea,electricity,gas,2025,gas (CC CCS),1.000000
141,None Specified,Enhanced-Ambition,South Korea,electricity,gas,2025,gas (CC H2 blend 50%),1.000000
142,None Specified,Enhanced-Ambition,South Korea,electricity,gas,2025,gas (CC),1.000000
143,None Specified,Enhanced-Ambition,South Korea,electricity,gas,2025,gas (steam/CT),0.251066
205,None Specified,Enhanced-Ambition,South Korea,electricity,geothermal,2025,geothermal,1.000000


In [62]:
df = conn.runQuery(queries[11], scenarios=scenarios[-2:], regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df['vint'] = df['technology'].str.split('=').str[1].astype(int)
df.head()

,Units,scenario,region,sector,subsector,technology,output,Year,value,vint
0,EJ,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2020",elect_td_bld,2020,0.031331,2020
1,EJ,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2025",elect_td_bld,2025,0.043087,2025
2,EJ,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2030",elect_td_bld,2030,0.055309,2030
3,EJ,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2035",elect_td_bld,2035,0.095608,2035
4,EJ,Enhanced-Ambition,South Korea,electricity,biomass,"biomass (IGCC) (dry cooling),year=2025",elec_biomass (IGCC),2025,0.000017,2025


In [63]:
df[(df['scenario'] == 'Current-Policy') & (df['Year'] == 2035) & (df['subsector'] == 'refined liquids')]#['value'].sum()

,Units,scenario,region,sector,subsector,technology,output,Year,value,vint


In [64]:
df[(df['Year'] == 2035)]

,Units,scenario,region,sector,subsector,technology,output,Year,value,vint
3,EJ,Enhanced-Ambition,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2035",elect_td_bld,2035,0.095608,2035
6,EJ,Enhanced-Ambition,South Korea,electricity,biomass,"biomass (IGCC) (dry cooling),year=2025",elec_biomass (IGCC),2035,0.000014,2025
8,EJ,Enhanced-Ambition,South Korea,electricity,biomass,"biomass (IGCC) (dry cooling),year=2030",elec_biomass (IGCC),2035,0.000041,2030
9,EJ,Enhanced-Ambition,South Korea,electricity,biomass,"biomass (IGCC) (dry cooling),year=2035",elec_biomass (IGCC),2035,0.000113,2035
12,EJ,Enhanced-Ambition,South Korea,electricity,biomass,"biomass (IGCC) (recirculating),year=2025",elec_biomass (IGCC),2035,0.000508,2025
...,...,...,...,...,...,...,...,...,...,...
474,EJ,Enhanced-Ambition,South Korea,electricity,wind,"wind_offshore,year=2035",electricity,2035,0.249926,2035
478,EJ,Enhanced-Ambition,South Korea,electricity,wind,"wind_storage,year=2020",electricity,2035,0.000208,2020
481,EJ,Enhanced-Ambition,South Korea,electricity,wind,"wind_storage,year=2025",electricity,2035,0.000215,2025
483,EJ,Enhanced-Ambition,South Korea,electricity,wind,"wind_storage,year=2030",electricity,2035,0.000532,2030


In [65]:
0.309-0.133

0.176

In [66]:
0.176 / 0.43

0.40930232558139534

In [67]:
df[(df['subsector'] == 'gas') & (df['output'].isin(['elec_gas (CC)', 'elec_gas (steam/CT)']))].groupby(['Year', 'scenario', 'vint'])['value'].sum()

Year  scenario           vint
1990  Enhanced-Ambition  1990    0.034576
2005  Enhanced-Ambition  2005    0.213065
2010  Enhanced-Ambition  2010    0.365474
2015  Enhanced-Ambition  2015    0.433323
2020  Enhanced-Ambition  2015    0.422126
                         2020    0.103874
2025  Enhanced-Ambition  2015    0.428242
                         2020    0.102824
                         2025    0.031934
2030  Enhanced-Ambition  2015    0.404304
                         2020    0.102273
                         2025    0.031461
                         2030    0.015946
2035  Enhanced-Ambition  2015    0.137318
                         2020    0.103009
                         2025    0.031678
                         2030    0.015824
                         2035    0.021164
Name: value, dtype: float64

In [68]:
0.156 / 0.433323

0.36000858482009956

In [69]:
0.01 / 0.433323

0.023077473385903817

In [70]:
df = conn.runQuery(queries[9], scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df.head()

,Units,scenario,region,subsector,technology,output,Year,value
0,EJ,Enhanced-Ambition,South Korea,biomass,biomass (IGCC),electricity,2025,0.000779
1,EJ,Enhanced-Ambition,South Korea,biomass,biomass (IGCC),electricity,2030,0.002863
2,EJ,Enhanced-Ambition,South Korea,biomass,biomass (IGCC),electricity,2035,0.007457
3,EJ,Enhanced-Ambition,South Korea,biomass,biomass (conv),electricity,2005,0.000162
4,EJ,Enhanced-Ambition,South Korea,biomass,biomass (conv),electricity,2010,0.001246


In [71]:
df[(df['subsector'] == 'refined liquids')]

,Units,scenario,region,subsector,technology,output,Year,value
66,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2020,0.002400
67,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2025,0.005712
68,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2030,0.014224
69,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2035,0.011738
70,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,1990,0.067888
71,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2005,0.075750
72,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2010,0.053866
73,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2015,0.038083
74,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2020,0.019856
75,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2025,0.025698


In [72]:
df[(df['Year'] == 2035) & (df['subsector'] == 'solar')]#.groupby(['scenario'])['value'].sum()

,Units,scenario,region,subsector,technology,output,Year,value
89,EJ,Enhanced-Ambition,South Korea,solar,PV,electricity,2035,0.463369
93,EJ,Enhanced-Ambition,South Korea,solar,PV_storage,electricity,2035,0.003233


In [73]:
282132 - 141804 + 282132

422460

In [74]:
df[(df['subsector'] == 'gas')]

,Units,scenario,region,subsector,technology,output,Year,value
24,EJ,Enhanced-Ambition,South Korea,gas,gas (CC CCS),electricity,2025,0.000812
25,EJ,Enhanced-Ambition,South Korea,gas,gas (CC CCS),electricity,2030,0.006633
26,EJ,Enhanced-Ambition,South Korea,gas,gas (CC CCS),electricity,2035,0.021174
27,EJ,Enhanced-Ambition,South Korea,gas,gas (CC H2 blend 50%),electricity,2025,0.014000
28,EJ,Enhanced-Ambition,South Korea,gas,gas (CC H2 blend 50%),electricity,2030,0.052007
29,EJ,Enhanced-Ambition,South Korea,gas,gas (CC H2 blend 50%),electricity,2035,0.110980
30,EJ,Enhanced-Ambition,South Korea,gas,gas (CC),electricity,1990,0.026982
31,EJ,Enhanced-Ambition,South Korea,gas,gas (CC),electricity,2005,0.202728
32,EJ,Enhanced-Ambition,South Korea,gas,gas (CC),electricity,2010,0.264734
33,EJ,Enhanced-Ambition,South Korea,gas,gas (CC),electricity,2015,0.429704


In [75]:
df[(df['subsector'] == 'hydro')]

,Units,scenario,region,subsector,technology,output,Year,value
46,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,1990,0.022900
47,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,2005,0.013223
48,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,2010,0.013255
49,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,2015,0.009511
50,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,2020,0.025700
51,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,2025,0.028800
52,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,2030,0.032800
53,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,2035,0.060100


In [76]:
df[(df['technology'].isin(['PV_storage', 'wind_storage']))].groupby(['scenario', 'Year'])['value'].sum()

scenario           Year
Enhanced-Ambition  2020    0.000220
                   2025    0.000482
                   2030    0.001393
                   2035    0.005917
Name: value, dtype: float64

In [77]:
df[(df['technology'].isin(['refined liquids (CC CCS)', 'refined liquids (CC)', 'refined liquids (steam/CT)']))]#.groupby(['scenario', 'Year'])['value'].sum()

,Units,scenario,region,subsector,technology,output,Year,value
66,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2020,0.002400
67,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2025,0.005712
68,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2030,0.014224
69,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2035,0.011738
70,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,1990,0.067888
71,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2005,0.075750
72,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2010,0.053866
73,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2015,0.038083
74,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2020,0.019856
75,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2025,0.025698


In [78]:
def catTech(x):
    if x in ['PV', 'PV_storage']:
        return 'Solar'
    elif x in ['wind', 'wind_offshore', 'wind_storage']:
        return 'Wind'
    elif x in ['refined liquids (CC)', 'refined liquids (steam/CT)']:
        return 'Oil'
    elif x in ['refined liquids (CC CCS)']:
        return 'Oil w/ CCS'
    elif x in ['hydro']:
        return 'Hydro'
    elif x in ['Gen_III', 'Gen_II_LWR']:
        return 'Nuclear'
    elif x in ['biomass (IGCC CCS)', 'biomass (conv CCS)']:
        return 'Biomass w/ CCS'
    elif x in ['biomass (IGCC)', 'biomass (conv)']:
        return 'Biomass'
    elif x in ['coal (conv pul ammonia blend 20%)']:
        return 'Ammonia'
    elif x in ['gas (CC H2 blend 50%)']:
        return 'Hydrogen'
    elif x in ['gas (CC CCS)']:
        return 'Gas w/ CCS'
    elif x in ['gas (CC)', 'gas (steam/CT)']:
        return 'Gas'
    elif x in ['coal (IGCC CCS)', 'coal (conv pul CCS)']:
        return 'Coal w/ CCS'
    elif x in ['coal (IGCC)', 'coal (conv pul)']:
        return 'Coal'

In [79]:
stack_order = [
    'Ammonia', 'Coal w/ CCS', 'Coal', 'Oil w/ CCS', 'Oil', 'Gas w/ CCS', 
    'Hydrogen', 'Gas',  'Nuclear', 'Biomass', 'Hydro', 'Wind', 'Solar'
]

In [80]:
df['genTech'] = df['technology'].apply(catTech)
df['genTech'] = pd.Categorical(df['genTech'], categories=stack_order, ordered=True)
df = df.sort_values(by=['Year', 'genTech'])
df['genTech'].unique()

['Coal', 'Oil', 'Gas', 'Nuclear', 'Hydro', ..., 'Wind', NaN, 'Coal w/ CCS', 'Gas w/ CCS', 'Hydrogen']
Length: 12
Categories (13, object): ['Ammonia' < 'Coal w/ CCS' < 'Coal' < 'Oil w/ CCS' ... 'Biomass' < 'Hydro' < 'Wind' < 'Solar']

In [81]:
df[(df['genTech']=="Oil")]

,Units,scenario,region,subsector,technology,output,Year,value,genTech
70,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,1990,0.067888,Oil
71,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2005,0.075750,Oil
72,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2010,0.053866,Oil
73,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2015,0.038083,Oil
66,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2020,0.002400,Oil
74,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2020,0.019856,Oil
67,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2025,0.005712,Oil
75,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2025,0.025698,Oil
68,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (CC),electricity,2030,0.014224,Oil
76,EJ,Enhanced-Ambition,South Korea,refined liquids,refined liquids (steam/CT),electricity,2030,0.024777,Oil


In [82]:
df[(df['Year'] == 2035) & (df['genTech'] == 'Coal w/ CCS')]

,Units,scenario,region,subsector,technology,output,Year,value,genTech
12,EJ,Enhanced-Ambition,South Korea,coal,coal (IGCC CCS),electricity,2035,0.020191,Coal w/ CCS
15,EJ,Enhanced-Ambition,South Korea,coal,coal (conv pul CCS),electricity,2035,0.102301,Coal w/ CCS


In [83]:
df['value'] = df['value'].apply(ej_to_twh)
df['Units'] = 'TWh'

In [84]:
df[(df['technology'] == 'gas (CC H2 blend 50%)')]

,Units,scenario,region,subsector,technology,output,Year,value,genTech
27,TWh,Enhanced-Ambition,South Korea,gas,gas (CC H2 blend 50%),electricity,2025,3.888889,Hydrogen
28,TWh,Enhanced-Ambition,South Korea,gas,gas (CC H2 blend 50%),electricity,2030,14.446389,Hydrogen
29,TWh,Enhanced-Ambition,South Korea,gas,gas (CC H2 blend 50%),electricity,2035,30.827889,Hydrogen


In [85]:
df = df.sort_values(by=['Year', 'genTech'], ascending=True)

In [86]:
df[(df['genTech'] == 'Ammonia') & (df['Year'] == 2035)]

,Units,scenario,region,subsector,technology,output,Year,value,genTech


In [87]:
df[(df['genTech'] == 'Coal w/ CCS') & (df['Year'] == 2035)]

,Units,scenario,region,subsector,technology,output,Year,value,genTech
12,TWh,Enhanced-Ambition,South Korea,coal,coal (IGCC CCS),electricity,2035,5.608722,Coal w/ CCS
15,TWh,Enhanced-Ambition,South Korea,coal,coal (conv pul CCS),electricity,2035,28.416944,Coal w/ CCS


In [88]:
df[(df['genTech'] == 'Others') & (df['Year'] == 2035)]

,Units,scenario,region,subsector,technology,output,Year,value,genTech


In [89]:
dfFig = df[(df['Year'] >= 2015) & (df['Year'] <= 2035) &  (~df['genTech'].isna())].groupby(['scenario', 'Year', 'genTech'], observed=False)['value'].sum().reset_index()
dfFig

,scenario,Year,genTech,value
0,Enhanced-Ambition,2015,Ammonia,0.000000
1,Enhanced-Ambition,2015,Coal w/ CCS,0.000000
2,Enhanced-Ambition,2015,Coal,215.851667
3,Enhanced-Ambition,2015,Oil w/ CCS,0.000000
4,Enhanced-Ambition,2015,Oil,10.578611
...,...,...,...,...
60,Enhanced-Ambition,2035,Nuclear,235.985834
61,Enhanced-Ambition,2035,Biomass,7.686964
62,Enhanced-Ambition,2035,Hydro,16.694444
63,Enhanced-Ambition,2035,Wind,158.473350


In [90]:
# Pivot for easier calculation
pivot = dfFig.pivot_table(index=['scenario', 'Year'], columns='genTech', values='value', fill_value=0)

# Apply multipliers
pivot['Hydrogen'] = pivot['Hydrogen'] * 0.5
pivot['Ammonia'] = pivot['Ammonia'] * 0.2

# Add "remains" to Gas and Coal
pivot['Gas'] += pivot['Hydrogen']/0.5 * 0.5  # (original Hydrogen value * 0.5)
pivot['Coal'] += pivot['Ammonia']/0.2 * 0.8  # (original Ammonia value * 0.8)

# If needed, you can reset index
result_df = pivot.reset_index().melt(id_vars=['scenario', 'Year'], var_name='genTech', value_name='value')
result_df

/tmp/ipykernel_105931/1785598069.py:2: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



,scenario,Year,genTech,value
0,Enhanced-Ambition,2015,Ammonia,0.000000
1,Enhanced-Ambition,2020,Ammonia,0.000000
2,Enhanced-Ambition,2025,Ammonia,0.000000
3,Enhanced-Ambition,2030,Ammonia,0.000000
4,Enhanced-Ambition,2035,Ammonia,0.000000
...,...,...,...,...
60,Enhanced-Ambition,2015,Solar,3.972611
61,Enhanced-Ambition,2020,Solar,22.202615
62,Enhanced-Ambition,2025,Solar,45.944245
63,Enhanced-Ambition,2030,Solar,86.201831


In [91]:
# dfAgg1 = result_df[(result_df['scenario'] == scenarios[0])]
# fig1 = px.bar(dfAgg1, x="Year", y="value", color="genTech", title="Current Policy", color_discrete_map=custom_colors)
dfAgg2 = result_df[(result_df['scenario'] == scenarios[-1])]
fig2 = px.bar(dfAgg2, x="Year", y="value", color="genTech", title="Enhanced Ambition", color_discrete_map=custom_colors)

In [92]:
result_df['RE'] = result_df['genTech'].apply(lambda x: 1 if x in ['Solar', 'Wind', 'Hydro', 'Biomass'] else 0)
result_df['CF'] = result_df['genTech'].apply(lambda x: 1 if x in ['Solar', 'Wind', 'Hydro', 'Biomass', 'Nuclear'] else 0)
# re_share_current = (result_df[(result_df['RE'] == 1) & (result_df['scenario'] == scenarios[0])].groupby(['Year'])['value'].sum() / result_df[(result_df['scenario'] == scenarios[0])].groupby(['Year'])['value'].sum()) * 100
# cf_share_current = (result_df[(result_df['CF'] == 1) & (result_df['scenario'] == scenarios[0])].groupby(['Year'])['value'].sum() / result_df[(result_df['scenario'] == scenarios[0])].groupby(['Year'])['value'].sum()) * 100
re_share_enhanced = (result_df[(result_df['RE'] == 1) & (result_df['scenario'] == scenarios[-1])].groupby(['Year'])['value'].sum() / result_df[(result_df['scenario'] == scenarios[-1])].groupby(['Year'])['value'].sum()) * 100
cf_share_enhanced = (result_df[(result_df['CF'] == 1) & (result_df['scenario'] == scenarios[-1])].groupby(['Year'])['value'].sum() / result_df[(result_df['scenario'] == scenarios[-1])].groupby(['Year'])['value'].sum()) * 100

In [93]:
years = list(range(2015, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# # Add traces from fig1 to col 1
# for trace in fig1['data']:
#     fig.add_trace(trace, row=1, col=1, secondary_y=False)
# for trace in fig1.data:
#     trace.showlegend = False
#     fig.add_trace(trace, row=1, col=1, secondary_y=False)
# # Add RE/carbon-free share line to col 1
# fig.add_trace(go.Scatter(
#     x=years,
#     y=re_share_current,  # % values
#     name="RE Share",
#     mode="markers",
#     marker=dict(symbol='triangle-up', size=9, color="green"),
#     showlegend=False
# ), row=1, col=1, secondary_y=True)

# fig.add_trace(go.Scatter(
#     x=years,
#     y=cf_share_current,  # % values
#     name="Carbon-Free Share",
#     mode="markers",
#     marker=dict(symbol='triangle-up', size=9, color="blue"), showlegend=False
# ), row=1, col=1, secondary_y=True)

# Add traces from fig5 to col 2
for trace in fig2['data']:
    fig.add_trace(trace, row=1, col=2, secondary_y=False)

# === Legend Group: Gases ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Technology</b>', showlegend=True, hoverinfo='skip'
))

# Add RE/carbon-free share to col 2
fig.add_trace(go.Scatter(
    x=years,
    y=re_share_enhanced,
    name="RE Share (%)",
    mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="green"),
), row=1, col=2, secondary_y=True)

fig.add_trace(go.Scatter(
    x=years,
    y=cf_share_enhanced,
    name="Carbon-Free Share (%)",
    mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="blue")
), row=1, col=2, secondary_y=True)

fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<b>Share</b>', showlegend=True, hoverinfo='skip'
))

fig.update_layout(
    yaxis=dict(title="Electricity Generation (TWh)", showgrid=True),
    yaxis1=dict(title="Electricity Generation (TWh)", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)

# for i, year in enumerate(range(2015, 2036, 5)):
    
#     fig.add_annotation(
#         x=years[i],
#         y=re_share_current[year] + 3,
#         xref="x1", yref="y2",
#         text=f"{re_share_current[year]:.0f}%",
#         showarrow=False,
#         arrowhead=2,
#         ax=0, ay=-30,
#         font=dict(color="green")
#     )

#     fig.add_annotation(
#         x=years[i],
#         y=cf_share_current[year] + 3,
#         xref="x1", yref="y2",
#         text=f"{cf_share_current[year]:.0f}%",
#         showarrow=False,
#         arrowhead=2,
#         ax=0, ay=-30,
#         font=dict(color="blue")
#     )

#     fig.add_annotation(
#         x=years[i],
#         y=re_share_enhanced[year] + 3,
#         xref="x2", yref="y4",
#         text=f"{re_share_enhanced[year]:.0f}%",
#         showarrow=False,
#         arrowhead=2,
#         ax=0, ay=-30,
#         font=dict(color="green")
#     )

#     fig.add_annotation(
#         x=years[i],
#         y=cf_share_enhanced[year] + 3,
#         xref="x2", yref="y4",
#         text=f"{cf_share_enhanced[year]:.0f}%",
#         showarrow=False,
#         arrowhead=2,
#         ax=0, ay=-30,
#         font=dict(color="blue")
#     )

fig.update_xaxes(tickangle=45)

fig.update_layout(
    yaxis=dict(title="Electricity Generation (TWh)", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Electricity Generation and Clean Energy Share</b>",
        font=dict(size=20),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=13),
        x=1.02, y=1,
        borderwidth=0
    )
)

for i in range(1, 3):
    fig.update_xaxes(
        tickmode='array',
        tickangle=45,
        tickvals=list(range(2015, 2040, 5)),
        tickfont=dict(size=15),
        row=1, col=i
    )

fig.update_layout(
    yaxis=dict(
        title="Electricity Generation (TWh)",
        title_font=dict(size=18)  # 👈 controls y-axis label size
    )
)

fig.show()


In [43]:
448.7 / 691.5

0.648879248011569

In [44]:
(179.9+24.3) / 691.5

0.2953000723065799